# Classification OD

Ce notebook permet la création de différent regroupement origine-destination dans Amsterdam soit à partir de découpages existants, soit en exploitant des méthodes d'apprentissage automatique. A partir de ces classifications est réalisée une petite analyse exploratoire afin de déterminer les différences entre les demandes Matsim_7-9 et AmsterdamCity_7-9 en ce qui concerne ces regroupements de couples origine-destination. Certaines classes seront alors sélectionnées pour l'étape de variation via classification.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade os
%pip install --upgrade sys

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade time

%pip install --upgrade matplotlib.pyplot

%pip install --upgrade numpy
%pip install --upgrade pandas
%pip install --upgrade geopandas

%pip install --plotly

%pip install --upgrade contextily

%pip install --upgrade shapely

%pip install --upgrade sklearn

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import os
import sys

from pathlib import Path

import importlib

import time

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import contextily as cx

from shapely.geometry import Point
from shapely import wkt
from shapely.ops import unary_union
from shapely.geometry import shape

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes

In [ ]:
# chargement des classes

import modules.Utils as Utils
import modules.ClassificationUtils as ClassificationUtils

importlib.reload(Utils)
importlib.reload(ClassificationUtils)

## 2 - Contexte, chargement, prétraitement

### 2.1 - Contexte

Une autre approche pour l'analyse de sensibilité d'hypothèses de construction de fichier de demande MnMS est de s'intéresser à leur distribution spatial. Pour rendre compte des différences en la matière sont construits dans ce notebook un ensemble de classification spatial. Une classification spatial est ici un certain nombre de regroupements de couples origine-destination, chaque individu de la demande étant associé à un seul de ces regroupements. Une classification spatiale est le résultat du produit cartésien d'un découpage en zone d'une surface donnée avec lui-même. L'origine du découpage peut-être administratif, données d'institution gouvernante, ou statistiques, produit par l'apprentissage automatique. 

### 2.2 - Chargement et prétraitement

In [ ]:

demand_area = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp')
#demand_area = gpd.read_file("../../data/AmsterdamCity/demand_areas/demand_areas.shp")

global_demand_area = unary_union(demand_area.geometry)

# augmentation de la surface 
distance=100
global_demand_area = shape(global_demand_area)
global_demand_area = shape(global_demand_area)
global_demand_area = global_demand_area.buffer( distance )

## 3 - Classification

L'objectif de cette sous-partie est de déterminer un ensemble de regroupement de couples OD. Quelque soit la méthode choisie, les regroupements reposeront sur le découpage par code postal des Pays-bas, le but étant d'obtenir des ensemble d'aggrégation de zones postales pour ensuite construire une classification par origine/destination en exploitant le produit cartésien de ces ensembles. Dans les parties suivantes deux paradigmes seront explorés : dans un premier des données en provenance de la municipalité d'Amsterdam à propos de découpages geographiques plus ou moins précis seront exploités et dans un deuxième des méthodes de classification automatique seront utilisées pour construire des découpages.

### 3.1 - Classification par découpage existants

#### 3.1.1 - Données

In [ ]:
# chargement des données

areas = gpd.read_file("../../data/AmsterdamCity/VMA/VMA30_areas.shp")

In [ ]:
# variables

areas.columns

Avec traduction cela donne :
- AREANR : identifiant de zone;
- NAME : nom;
- CENTROIDNR : numéro de centroïd;
- STADSDELEN : division administrative officielle/ arrondissement;
- VENOM_ZONE : zone VENOM (Verkeer en Netwerk Optimalisatie Model, modèle de trafic utilisé au Pays-bas);
- BUURT : quartier (plus petite unité administrative des Pays-bas);
- GEBIED_ACT : "zone opérationelle actuelle" (découpage utilisé pour des études statistiques);
- GEMEENTE : municipalité (zone administrative);
- PLAATS : localité;
- BUURT_NAAM : nom de quartier;
- SCHADUWPRO : profil d'ombre, 0 si les données spatiales sont exactes/ certifiées, 1 dans le cas contraire;
- OPP : surface;
- OPPER : superficie;
- geometry : données géospatiales.

In [ ]:
# valeurs uniques

print(f'Nombre de lignes : {len(areas)}.')
print(f'Identifiants de zone : {areas["AREANR"].nunique()}.')
print(f'noms : {areas["NAME"].nunique()}.')
print(f'Numéros de centroïd : {areas["CENTROIDNR"].nunique()}.')
print(f'Arrondissements : {areas["STADSDELEN"].nunique()}.')
print(f'Zones VENOM : {areas["VENOM_ZONE"].nunique()}.')
print(f'Quartiers : {areas["BUURT"].nunique()}.')
print(f'Zones opérationelles actuelles : {areas["GEBIED_ACT"].nunique()}.')
print(f'Municipalités : {areas["GEMEENTE"].nunique()}.')
print(f'Localités : {areas["PLAATS"].nunique()}.')
print(f'Noms de quartier : {areas["BUURT_NAAM"].nunique()}.')
print(f'Profils d ombre : {areas["SCHADUWPRO"].nunique()}.')
print(f'Surface : {areas["OPP"].nunique()}.')
print(f'Superficie : {areas["OPPER"].nunique()}.')
print(f'Données géospatiales : {areas["geometry"].nunique()}.')

Les variables noms, Numéros de centroid, Profils d'ombre et surface ont toutes leurs valeurs à None ou NaN.
En se basant uniquement sur les nombres de valeurs uniques, ci-dessous la liste des découpages spatiales du plus au moins aggrégé : 
- Zones opérationelles actuelles (9);
- Arrondissements (10);
- Municipalités (404);
- Quartiers (500);
- Localités (997);
- Noms de quartier (3718);
- Zones Venom (3807);
- Identifiant de zones (4881)

In [ ]:
# changement de nom de variables

areas = areas.rename(columns={"AREANR":"ZONE_ID","CENTROIDNR":"CENTROID_ID","STADSDELEN":"DISTRICT","BUURT":"NEIGHBOURHOOD","GEBIED_ACT":"STAT_ZONE","GEMEENTE":"MUNICIPALITY","PLAATS":"LOCALITY","BUURT_NAAM":"NEIGHBOURHOOD_NAME","SCHADUWPRO":"SHADOW_PROFILE","OPP":"AREA","OPPER":"SURFACE_AREA","geometry":"GEOMETRY"})
areas = gpd.GeoDataFrame(areas, geometry="GEOMETRY", crs="epsg:28992")

In [ ]:
areas.columns

In [ ]:
# affichage des zones opérationnelles actuelles

stat_zone = areas.dissolve(by="STAT_ZONE").reset_index()
stat_zone.plot()

In [ ]:
# non prise en compte de STAT_ZONE == 9 car il s'agit d'une gigantesque zone recouvrant une grande partie de l'Europe de l'ouest.
stat_zone.loc[stat_zone["STAT_ZONE"]!="9"].plot()

In [ ]:
# affichage des arrondissements 

district = areas.dissolve(by="DISTRICT").reset_index()
district.plot()

In [ ]:
# non prise en compte de DISTRICT == Studiegebied_VENOM car il s'agit d'une gigantesque zone recouvrant une grande partie de l'Europe de l'ouest.

district.loc[district["DISTRICT"]!="Studiegebied_VENOM"].plot()

In [ ]:
# decoupage par quartier restreint à la zone de demande

district_filtered = district.loc[district["DISTRICT"]!="Studiegebied_VENOM"]
district_filtered = district_filtered[district_filtered.geometry.intersects(global_demand_area)]
gdf_global = gpd.GeoDataFrame(geometry=[global_demand_area], crs="epsg:28992")
print(f"Nombre de zones : {len(district_filtered)}.")
ax = district_filtered.plot()
gdf_global.plot(ax=ax, facecolor='none', edgecolor='black')

Le découpage en zones actuelles opérationelles et par arrondissement sont exactement simialires à l'exception de de l'ajout d'une zone pour le découpage par arrondissement.

In [ ]:
# affichage des municipalités 

municipality = areas.dissolve(by="MUNICIPALITY").reset_index()
municipality.plot()

In [ ]:
# non prise en compte de MUNICIPALITY == Buitenland car il s'agit d'une gigantesque zone recouvrant une grande partie de l'Europe de l'ouest.
municipality.loc[municipality["MUNICIPALITY"]!="Buitenland"].plot()

In [ ]:
# decoupage par municipalité restreint à la zone de demande

municipality_filtered = municipality[municipality.geometry.intersects(global_demand_area)]
gdf_global = gpd.GeoDataFrame(geometry=[global_demand_area], crs="epsg:28992")
print(f"Nombre de zones : {len(municipality_filtered)}.")
ax = municipality_filtered.plot()
gdf_global.plot(ax=ax, facecolor='none', edgecolor='black')

In [ ]:
# affichage des quartiers 

neighbourhood = areas.dissolve(by="NEIGHBOURHOOD").reset_index()
neighbourhood.plot()

In [ ]:
# affichage des quartiers 
# non prise en compte de NEIGHBOURHOOD == Z99z car il s'agit d'une gigantesque zone recouvrant une grande partie de l'Europe de l'ouest.
neighbourhood.loc[neighbourhood["NEIGHBOURHOOD"]!="Z99z"].plot()

In [ ]:
# decoupage par quartier restreint à la zone de demande

neighbourhood_filtered = neighbourhood.loc[neighbourhood["NEIGHBOURHOOD"]!="Z99z"]
neighbourhood_filtered = neighbourhood_filtered[neighbourhood_filtered.geometry.intersects(global_demand_area)]
gdf_global = gpd.GeoDataFrame(geometry=[global_demand_area], crs="epsg:28992")
print(f"Nombre de zones : {len(neighbourhood_filtered)}.")
ax = neighbourhood_filtered.plot()
gdf_global.plot(ax=ax, facecolor='none', edgecolor='black')

In [ ]:
# decoupage par quartier restreint à la zone de demande

neighbourhood_filtered = neighbourhood.loc[neighbourhood["NEIGHBOURHOOD"]!="Z99z"]
neighbourhood_filtered = neighbourhood_filtered[neighbourhood_filtered.geometry.within(global_demand_area)]
gdf_global = gpd.GeoDataFrame(geometry=[global_demand_area], crs="epsg:28992")
print(f"Nombre de zones : {len(neighbourhood_filtered)} + 3.")
ax = neighbourhood_filtered.plot()
gdf_global.plot(ax=ax, facecolor='none', edgecolor='black')

#### 3.1.1 - Bilan

Au vu des différents découpages spatiaux, il semblerait qu'il n'y ait qu'un seul choix pertinent : le découpage par discrict. En considérant les points d'entrées et de sorties comme des zones à part entière, le découpage finale par catégories existantes contient 15 zones. 

In [ ]:
# affichage des zones
list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]

areas = gpd.read_file("../../data/AmsterdamCity/VMA/VMA30_areas.shp")
areas = areas.rename(columns={"AREANR":"ZONE_ID","CENTROIDNR":"CENTROID_ID","STADSDELEN":"DISTRICT","BUURT":"NEIGHBOURHOOD","GEBIED_ACT":"STAT_ZONE","GEMEENTE":"MUNICIPALITY","PLAATS":"LOCALITY","BUURT_NAAM":"NEIGHBOURHOOD_NAME","SCHADUWPRO":"SHADOW_PROFILE","OPP":"AREA","OPPER":"SURFACE_AREA","geometry":"GEOMETRY"})
areas = gpd.GeoDataFrame(areas, geometry="GEOMETRY", crs="epsg:28992")


district = areas.dissolve(by="DISTRICT").reset_index()

district_filtered = district.loc[district["DISTRICT"]!="Studiegebied_VENOM"]
district_filtered = district_filtered[district_filtered.geometry.intersects(global_demand_area)]


# Création du GeoDataFrame des points
gdf_points = gpd.GeoDataFrame(
    geometry=[Point(xy) for xy in zip(list_long, list_lat)],
    crs="EPSG:4326"
)
# Reprojection si nécessaire
gdf_points = gdf_points.to_crs("EPSG:28992")


district_clipped = gpd.overlay(
    district_filtered, 
    gdf_global, 
    how="intersection"
)


import contextily as cx
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from shapely.geometry import Point



#---------- 2. TRONQUER LES ZONES ----------
district_clipped = gpd.overlay(
    district_filtered,
    gdf_global,
    how="intersection"
)

# Nombre total de clusters = zones + points
total_clusters = len(district_clipped) + len(gdf_points)

colors = plt.cm.tab20(np.linspace(0, 1, total_clusters))


#---------- 3. REPROJECTER POUR LE FOND DE CARTE ----------
district_web = district_clipped.to_crs(epsg=3857)
points_web = gdf_points.to_crs(epsg=3857)
global_web = gdf_global.to_crs(epsg=3857)


#---------- 4. PLOT ----------
fig, ax = plt.subplots(figsize=(12, 12))

# Zones
for i, (_, row) in enumerate(district_web.iterrows()):
    geom = row.geometry
    
    # Dessiner la géométrie quel que soit son type (Polygon, MultiPolygon, LineString,…)
    gpd.GeoSeries([geom]).plot(
        ax=ax, 
        color=colors[i], 
        edgecolor="black", 
        linewidth=1, 
        alpha=0.6
    )
    
    # Étiquette si centroid disponible
    if geom.geom_type in ["Polygon", "MultiPolygon"]:
        Cx, Cy = geom.centroid.x, geom.centroid.y
        ax.text(Cx, Cy, f"{i}", fontsize=9, color="black")


# Points (avec continuation de la numérotation)
start = len(district_web)

for j, (idx, row) in enumerate(points_web.iterrows()):
    cluster_id = start + j
    px, py = row.geometry.x, row.geometry.y
    ax.scatter(px, py, color=colors[cluster_id], s=400, edgecolor="black")
    ax.text(px, py, f"{cluster_id}", fontsize=10, color="black", fontweight="bold")

# Polygone global
global_web.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1)


#---------- 5. FOND DE CARTE ----------
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

ax.set_axis_off()
plt.title("Clusters (zones + points) avec fond de carte et numérotation unifiée")
plt.show()


A partir de ces zones sont formées les groupes origine-destination, l'ensemble des groupes étant le produit cartésien du découpage spatial ci-dessus.

In [ ]:
# creation des clusters

#---------------------------------------
# 1. Fusion : zones + points en clusters
#---------------------------------------
gdf_zones = district_web.reset_index(drop=True).copy()
gdf_zones["CLUSTER_ID"] = gdf_zones.index

gdf_pts = points_web.reset_index(drop=True).copy()
gdf_pts["CLUSTER_ID"] = gdf_pts.index + len(gdf_zones)

gdf_clusters = pd.concat([gdf_zones, gdf_pts], ignore_index=True)
gdf_clusters = gpd.GeoDataFrame(gdf_clusters, geometry="geometry", crs="EPSG:3857")
gdf_clusters = gdf_clusters.to_crs("epsg:32631")

#---------------------------------------
# 2. Produit cartésien (cross-join)
#---------------------------------------
df_pairs = gdf_clusters.merge(
    gdf_clusters,
    how="cross",
    suffixes=("_origin", "_dest")
)


#---------------------------------------
# 3. Colonnes demandées
#---------------------------------------
df_pairs = df_pairs.rename(columns={
    "CLUSTER_ID_origin": "ORIGIN_CLUSTER",
    "CLUSTER_ID_dest": "DESTINATION_CLUSTER",
    "geometry_origin": "ORIGIN",
    "geometry_dest": "DESTINATION"
})

df_pairs["CLUSTER_ID"] = df_pairs.index


#---------------------------------------
# 4. Colonnes finales
#---------------------------------------
df_pairs = df_pairs[[
    "CLUSTER_ID",
    "ORIGIN_CLUSTER",
    "DESTINATION_CLUSTER",
    "ORIGIN",
    "DESTINATION"
]]


In [ ]:
# clusters

df_pairs
df_pairs.to_csv("../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv", sep=';', index=False)

### 3.2 - Classification automatique

Il s'agit ici d'exploiter des techniques d'apprentissage non supervisée afin d'obtenir des regroupement de zones postales. Seules les variables relatives aux caractéristiques spatiales de chaque zone seront prises en compte afin d'obtenir des classes indépendantes des fichiers de demande. En pratique la classification supervisée sera effectuée sur les centroides de chaque zones postales, donc sur des critères de proximitées dans l'espace.

#### 3.2.1 - Prétraitement

Dans un premier temps, il s'agit de déterminer les centroides de chaque zone postale. Les codes ci-dessous sont ceux produit par .

In [ ]:
# chargement des zones postales correspondant à la zone de demande, calcul des centroides

gdf_areas = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp')

#gdf_areas = gpd.read_file("../../data/AmsterdamCity/demand_areas/demand_areas.shp")
gdf_areas = gdf_areas.to_crs('EPSG:3857')                        
s_centroids = gdf_areas.centroid
s_centroids_dist = s_centroids.to_crs('EPSG:28992') # for dist calculation

In [ ]:
if False:
    # Calcul de poids, accumulation totale (origine + destination)
    s_centroids_gps = s_centroids.to_crs('EPSG:4326') #WGS-84 for HERE API 
    df_centroids = pd.DataFrame({'id':gdf_areas['[(\'Deletio'],'lat':s_centroids_gps.x, 'long':s_centroids_gps.y,'weight':np.zeros(len(gdf_areas))})
    t1 = time.time()
    i_print=0

    data_dmd = pd.read_csv('../../data/AmsterdamCity/81_trips_prog_VMA_2019.csv', sep=';')
    nb_data_dmd = len(data_dmd)

    for index, row in data_dmd[:].iterrows():
        o_loc = str(int(row['O']))
        d_loc = str(int(row['D']))
        dmd_loc = row['OVOS_trips'] + row['PAOS_trips'] +  row['FIOS_trips']# PT (OV) and car (PA) and bike for morning peak (OS) 7-9am
        if o_loc in df_centroids['id'].values: #inner Ams
            df_centroids.loc[df_centroids['id']==o_loc,'weight'] += dmd_loc
        if d_loc in df_centroids['id'].values: #inner Ams
            df_centroids.loc[df_centroids['id']==d_loc,'weight'] += dmd_loc

        if index>i_print*nb_data_dmd//10:
            print(str(i_print*10)+'%')
            i_print += 1
    t2 = time.time()
    print(t2-t1, 's')

    # sauvegarde dans centroids_Ams
    df_centroids.to_csv('../../data/AmsterdamCity/centroids_Ams.csv')
    #df_centroids.to_csv('../../data/AmsterdamCity/centroids_demand.csv')

else:
    # chargement des centroids
    df_centroids = pd.read_csv('../../data/AmsterdamCity/centroids_Ams.csv')
    #df_centroids = pd.read_csv('../../data/AmsterdamCity/centroids_demand.csv')

#### 3.2.2 - K-means

explication de l'algo des K-means

##### 3.2.2.1 - K = 7

In [ ]:
# algorithme des K-means k = 7


nb_clusters = 7
X = [[x,y] for (x,y) in zip(s_centroids_dist.x, s_centroids_dist.y)]
kmeans = KMeans(n_clusters = nb_clusters, random_state=0).fit(X)#, sample_weight=df_centroids['weight'].values)
#gdf = gpd.GeoDataFrame(df_centroids, geometry=s_centroids)

gdf_areas['cluster'] = kmeans.labels_
df_centroids['cluster'] = kmeans.labels_

gdf_KMeans_7 = gdf_areas.dissolve(by='cluster')

In [ ]:
# indicateur

inertia = kmeans.inertia_
silhouette = silhouette_score(X, kmeans.labels_)
davies = davies_bouldin_score(X, kmeans.labels_)
calinski = calinski_harabasz_score(X, kmeans.labels_)

# Affichage propre
df_metrics = pd.DataFrame({
    "Indicateur": [
        "Inertie (within-cluster SSE)",
        "Silhouette score",
        "Davies-Bouldin index",
        "Calinski-Harabasz index"
    ],
    "Valeur": [
        inertia,
        silhouette,
        davies,
        calinski
    ]
})

print(df_metrics)

In [ ]:
# affichage des K-means avec k = 7

list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]
db_access = pd.DataFrame({'long':list_long, 'lat':list_lat})

gdf_access = gpd.GeoDataFrame(db_access, geometry = gpd.points_from_xy(list_long, list_lat))
gdf_access.crs = 'EPSG:4326'
gdf_access = gdf_access.to_crs('EPSG:3857')
gdf_access_dist = gdf_access.to_crs('EPSG:28992')

nb_points = len(db_access)

gdf_access['closest_centroid'] = np.zeros(nb_points)
gdf_access['i_closest_centroid'] = np.zeros(nb_points)
for i,row in gdf_access_dist.iterrows():
    x=row.geometry.x
    y=row.geometry.y
    dists = [(x-x_)**2+(y-y_)**2 for (x_,y_) in zip(s_centroids_dist.x, s_centroids_dist.y)]
    centroid_id = df_centroids['id'][np.argmin(dists)]
    gdf_access.loc[i, 'i_closest_centroid'] = np.argmin(dists)
    gdf_access.loc[i, 'closest_centroid'] = centroid_id


fig, ax=plt.subplots(figsize=(20,20))
#gdf_clusters = gdf_clusters.to_crs('EPSG:3857')
gdf_areas.centroid.plot(ax=ax, color='b')
gdf_KMeans_7.centroid.plot(ax=ax, color = 'r')
gdf_KMeans_7.reset_index().plot(
    ax=ax, column='cluster', cmap='rainbow', edgecolor='black', alpha=0.5, zorder=1)
#gdf_edges[gdf_edges.FuncClass<5].plot(ax=ax, color='grey')
gdf_access.plot(ax=ax, color='magenta', markersize=100)
cx.add_basemap(ax=ax, url=cx.providers.OpenStreetMap.Mapnik)
for c in range(nb_clusters):
    x = gdf_KMeans_7.centroid.loc[c].x
    y = gdf_KMeans_7.centroid.loc[c].y
    ax.text(x,y,str(c), zorder=2, size=40)
for p in range(nb_points):
    x = gdf_access.centroid.loc[p].x
    y = gdf_access.centroid.loc[p].y
    t = ax.text(x,y,str(p), zorder=2, size=40, color='m')
    t.set_bbox(dict(facecolor='w', alpha=0.5, edgecolor='w'))
    i_cen_id = int(gdf_access.loc[p, 'i_closest_centroid'])
    cl_cen = gdf_areas.centroid[i_cen_id]
    ax.plot(cl_cen.x, cl_cen.y, 'kd', alpha=0.5, markersize=20)

In [ ]:
# création de la classification OD

# Polygones des clusters (issus du dissolve)
gdf_polygons_K7 = gdf_KMeans_7.reset_index()[['cluster', 'geometry']]
gdf_polygons_K7 = gdf_polygons_K7.rename(columns={'cluster': 'CLUSTER_ID'})

gdf_polygons_K7 = gpd.GeoDataFrame(gdf_polygons_K7, geometry='geometry', crs = 'EPSG:3857')
gdf_polygons_K7 = gdf_polygons_K7.to_crs("epsg:32631")

gdf_points = gpd.GeoDataFrame(
    db_access.copy(), 
    geometry=gpd.points_from_xy(db_access.long, db_access.lat),
    crs="EPSG:4326"
)

# Reprojection dans le même CRS que les polygones
gdf_points = gdf_points.to_crs(gdf_polygons_K7.crs)

# Ajout identifiants
gdf_points = gdf_points.reset_index().rename(columns={'index': 'CLUSTER_ID'})
gdf_points = gdf_points[['CLUSTER_ID', 'geometry']]


clusters_OD_K7 = pd.concat([gdf_polygons_K7, gdf_points], ignore_index=True)
clusters_OD_K7 = gpd.GeoDataFrame(clusters_OD_K7, geometry='geometry', crs=gdf_polygons_K7.crs)

clusters_OD_K7 = clusters_OD_K7.merge(
    clusters_OD_K7,
    how="cross",
    suffixes=("_origin", "_dest")
)


#---------------------------------------
# 3. Colonnes demandées
#---------------------------------------
clusters_OD_K7 = clusters_OD_K7.rename(columns={
    "CLUSTER_ID_origin": "ORIGIN_CLUSTER",
    "CLUSTER_ID_dest": "DESTINATION_CLUSTER",
    "geometry_origin": "ORIGIN",
    "geometry_dest": "DESTINATION"
})

clusters_OD_K7["CLUSTER_ID"] = clusters_OD_K7.index


#---------------------------------------
# 4. Colonnes finales
#---------------------------------------
clusters_OD_K7 = clusters_OD_K7[[
    "CLUSTER_ID",
    "ORIGIN_CLUSTER",
    "DESTINATION_CLUSTER",
    "ORIGIN",
    "DESTINATION"
]]

clusters_OD_K7

clusters_OD_K7.to_csv("../../data/classification/KMeans/Classification_KMeans-7.csv", sep=';', index=False)

##### 3.2.2.2 - K = 10

In [ ]:
# algorithme des K-means k = 10

nb_clusters = 10
X = [[x,y] for (x,y) in zip(s_centroids_dist.x, s_centroids_dist.y)]
kmeans = KMeans(n_clusters = nb_clusters, random_state=0).fit(X)#, sample_weight=df_centroids['weight'].values)
#gdf = gpd.GeoDataFrame(df_centroids, geometry=s_centroids)

gdf_areas['cluster'] = kmeans.labels_
df_centroids['cluster'] = kmeans.labels_

gdf_KMeans_10 = gdf_areas.dissolve(by='cluster')

In [ ]:
# indicateur

inertia = kmeans.inertia_
silhouette = silhouette_score(X, kmeans.labels_)
davies = davies_bouldin_score(X, kmeans.labels_)
calinski = calinski_harabasz_score(X, kmeans.labels_)

# Affichage propre
df_metrics = pd.DataFrame({
    "Indicateur": [
        "Inertie (within-cluster SSE)",
        "Silhouette score",
        "Davies-Bouldin index",
        "Calinski-Harabasz index"
    ],
    "Valeur": [
        inertia,
        silhouette,
        davies,
        calinski
    ]
})

print(df_metrics)

In [ ]:
# affichage des K-means avec k = 10

list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]
db_access = pd.DataFrame({'long':list_long, 'lat':list_lat})

gdf_access = gpd.GeoDataFrame(db_access, geometry = gpd.points_from_xy(list_long, list_lat))
gdf_access.crs = 'EPSG:4326'
gdf_access = gdf_access.to_crs('EPSG:3857')
gdf_access_dist = gdf_access.to_crs('EPSG:28992')

nb_points = len(db_access)

gdf_access['closest_centroid'] = np.zeros(nb_points)
gdf_access['i_closest_centroid'] = np.zeros(nb_points)
for i,row in gdf_access_dist.iterrows():
    x=row.geometry.x
    y=row.geometry.y
    dists = [(x-x_)**2+(y-y_)**2 for (x_,y_) in zip(s_centroids_dist.x, s_centroids_dist.y)]
    centroid_id = df_centroids['id'][np.argmin(dists)]
    gdf_access.loc[i, 'i_closest_centroid'] = np.argmin(dists)
    gdf_access.loc[i, 'closest_centroid'] = centroid_id


fig, ax=plt.subplots(figsize=(20,20))
#gdf_clusters = gdf_clusters.to_crs('EPSG:3857')
gdf_areas.centroid.plot(ax=ax, color='b')
gdf_KMeans_10.centroid.plot(ax=ax, color = 'r')
gdf_KMeans_10.reset_index().plot(
    ax=ax, column='cluster', cmap='rainbow', edgecolor='black', alpha=0.5, zorder=1)
#gdf_edges[gdf_edges.FuncClass<5].plot(ax=ax, color='grey')
gdf_access.plot(ax=ax, color='magenta', markersize=100)
cx.add_basemap(ax=ax, url=cx.providers.OpenStreetMap.Mapnik)
for c in range(nb_clusters):
    x = gdf_KMeans_10.centroid.loc[c].x
    y = gdf_KMeans_10.centroid.loc[c].y
    ax.text(x,y,str(c), zorder=2, size=40)
for p in range(nb_points):
    x = gdf_access.centroid.loc[p].x
    y = gdf_access.centroid.loc[p].y
    t = ax.text(x,y,str(p), zorder=2, size=40, color='m')
    t.set_bbox(dict(facecolor='w', alpha=0.5, edgecolor='w'))
    i_cen_id = int(gdf_access.loc[p, 'i_closest_centroid'])
    cl_cen = gdf_areas.centroid[i_cen_id]
    ax.plot(cl_cen.x, cl_cen.y, 'kd', alpha=0.5, markersize=20)

In [ ]:
# création de la classification OD

# Polygones des clusters (issus du dissolve)
gdf_polygons_K10 = gdf_KMeans_10.reset_index()[['cluster', 'geometry']]
gdf_polygons_K10 = gdf_polygons_K10.rename(columns={'cluster': 'CLUSTER_ID'})

gdf_polygons_K10 = gpd.GeoDataFrame(gdf_polygons_K10, geometry='geometry', crs = 'EPSG:3857')
gdf_polygons_K10 = gdf_polygons_K7.to_crs("epsg:32631")

gdf_points = gpd.GeoDataFrame(
    db_access.copy(), 
    geometry=gpd.points_from_xy(db_access.long, db_access.lat),
    crs="EPSG:4326"
)

# Reprojection dans le même CRS que les polygones
gdf_points = gdf_points.to_crs(gdf_polygons_K10.crs)

# Ajout identifiants
gdf_points = gdf_points.reset_index().rename(columns={'index': 'CLUSTER_ID'})
gdf_points = gdf_points[['CLUSTER_ID', 'geometry']]


clusters_OD_K10 = pd.concat([gdf_polygons_K10, gdf_points], ignore_index=True)
clusters_OD_K10 = gpd.GeoDataFrame(clusters_OD_K10, geometry='geometry', crs=gdf_polygons_K10.crs)

clusters_OD_K10 = clusters_OD_K10.merge(
    clusters_OD_K10,
    how="cross",
    suffixes=("_origin", "_dest")
)


#---------------------------------------
# 3. Colonnes demandées
#---------------------------------------
clusters_OD_K10 = clusters_OD_K10.rename(columns={
    "CLUSTER_ID_origin": "ORIGIN_CLUSTER",
    "CLUSTER_ID_dest": "DESTINATION_CLUSTER",
    "geometry_origin": "ORIGIN",
    "geometry_dest": "DESTINATION"
})

clusters_OD_K10["CLUSTER_ID"] = clusters_OD_K10.index


#---------------------------------------
# 4. Colonnes finales
#---------------------------------------
clusters_OD_K10 = clusters_OD_K10[[
    "CLUSTER_ID",
    "ORIGIN_CLUSTER",
    "DESTINATION_CLUSTER",
    "ORIGIN",
    "DESTINATION"
]]

clusters_OD_K10

clusters_OD_K10.to_csv("../../data/classification/KMeans/Classification_KMeans-10.csv", sep=';', index=False)

##### 3.2.2.3 - K = 15

In [ ]:
# algorithme des K-means k = 15

nb_clusters = 15
X = [[x,y] for (x,y) in zip(s_centroids_dist.x, s_centroids_dist.y)]
kmeans = KMeans(n_clusters = nb_clusters, random_state=0).fit(X)#, sample_weight=df_centroids['weight'].values)
#gdf = gpd.GeoDataFrame(df_centroids, geometry=s_centroids)

gdf_areas['cluster'] = kmeans.labels_
df_centroids['cluster'] = kmeans.labels_

gdf_KMeans_15 = gdf_areas.dissolve(by='cluster')

In [ ]:
# indicateur

inertia = kmeans.inertia_
silhouette = silhouette_score(X, kmeans.labels_)
davies = davies_bouldin_score(X, kmeans.labels_)
calinski = calinski_harabasz_score(X, kmeans.labels_)

# Affichage propre
df_metrics = pd.DataFrame({
    "Indicateur": [
        "Inertie (within-cluster SSE)",
        "Silhouette score",
        "Davies-Bouldin index",
        "Calinski-Harabasz index"
    ],
    "Valeur": [
        inertia,
        silhouette,
        davies,
        calinski
    ]
})

print(df_metrics)

In [ ]:
# affichage des K-means avec k = 15

list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]
db_access = pd.DataFrame({'long':list_long, 'lat':list_lat})

gdf_access = gpd.GeoDataFrame(db_access, geometry = gpd.points_from_xy(list_long, list_lat))
gdf_access.crs = 'EPSG:4326'
gdf_access = gdf_access.to_crs('EPSG:3857')
gdf_access_dist = gdf_access.to_crs('EPSG:28992')

nb_points = len(db_access)

gdf_access['closest_centroid'] = np.zeros(nb_points)
gdf_access['i_closest_centroid'] = np.zeros(nb_points)
for i,row in gdf_access_dist.iterrows():
    x=row.geometry.x
    y=row.geometry.y
    dists = [(x-x_)**2+(y-y_)**2 for (x_,y_) in zip(s_centroids_dist.x, s_centroids_dist.y)]
    centroid_id = df_centroids['id'][np.argmin(dists)]
    gdf_access.loc[i, 'i_closest_centroid'] = np.argmin(dists)
    gdf_access.loc[i, 'closest_centroid'] = centroid_id


fig, ax=plt.subplots(figsize=(20,20))
#gdf_clusters = gdf_clusters.to_crs('EPSG:3857')
gdf_areas.centroid.plot(ax=ax, color='b')
gdf_KMeans_15.centroid.plot(ax=ax, color = 'r')
gdf_KMeans_15.reset_index().plot(
    ax=ax, column='cluster', cmap='rainbow', edgecolor='black', alpha=0.5, zorder=1)
#gdf_edges[gdf_edges.FuncClass<5].plot(ax=ax, color='grey')
gdf_access.plot(ax=ax, color='magenta', markersize=100)
cx.add_basemap(ax=ax, url=cx.providers.OpenStreetMap.Mapnik)
for c in range(nb_clusters):
    x = gdf_KMeans_15.centroid.loc[c].x
    y = gdf_KMeans_15.centroid.loc[c].y
    ax.text(x,y,str(c), zorder=2, size=40)
for p in range(nb_points):
    x = gdf_access.centroid.loc[p].x
    y = gdf_access.centroid.loc[p].y
    t = ax.text(x,y,str(p), zorder=2, size=40, color='m')
    t.set_bbox(dict(facecolor='w', alpha=0.5, edgecolor='w'))
    i_cen_id = int(gdf_access.loc[p, 'i_closest_centroid'])
    cl_cen = gdf_areas.centroid[i_cen_id]
    ax.plot(cl_cen.x, cl_cen.y, 'kd', alpha=0.5, markersize=20)

In [ ]:
# création de la classification OD

# Polygones des clusters (issus du dissolve)
gdf_polygons_K15 = gdf_KMeans_15.reset_index()[['cluster', 'geometry']]
gdf_polygons_K15 = gdf_polygons_K15.rename(columns={'cluster': 'CLUSTER_ID'})

gdf_polygons_K15 = gpd.GeoDataFrame(gdf_polygons_K15, geometry='geometry', crs = 'EPSG:3857')
gdf_polygons_K15 = gdf_polygons_K15.to_crs("epsg:32631")

gdf_points = gpd.GeoDataFrame(
    db_access.copy(), 
    geometry=gpd.points_from_xy(db_access.long, db_access.lat),
    crs="EPSG:4326"
)

# Reprojection dans le même CRS que les polygones
gdf_points = gdf_points.to_crs(gdf_polygons_K15.crs)

# Ajout identifiants
gdf_points = gdf_points.reset_index().rename(columns={'index': 'CLUSTER_ID'})
gdf_points = gdf_points[['CLUSTER_ID', 'geometry']]


clusters_OD_K15 = pd.concat([gdf_polygons_K15, gdf_points], ignore_index=True)
clusters_OD_K15 = gpd.GeoDataFrame(clusters_OD_K15, geometry='geometry', crs=gdf_polygons_K10.crs)

clusters_OD_K15 = clusters_OD_K15.merge(
    clusters_OD_K15,
    how="cross",
    suffixes=("_origin", "_dest")
)


#---------------------------------------
# 3. Colonnes demandées
#---------------------------------------
clusters_OD_K15 = clusters_OD_K15.rename(columns={
    "CLUSTER_ID_origin": "ORIGIN_CLUSTER",
    "CLUSTER_ID_dest": "DESTINATION_CLUSTER",
    "geometry_origin": "ORIGIN",
    "geometry_dest": "DESTINATION"
})

clusters_OD_K15["CLUSTER_ID"] = clusters_OD_K15.index


#---------------------------------------
# 4. Colonnes finales
#---------------------------------------
clusters_OD_K15 = clusters_OD_K15[[
    "CLUSTER_ID",
    "ORIGIN_CLUSTER",
    "DESTINATION_CLUSTER",
    "ORIGIN",
    "DESTINATION"
]]

clusters_OD_K15

clusters_OD_K15.to_csv("../../data/classification/KMeans/Classification_KMeans-15.csv", sep=';', index=False)

##### 3.2.2.4 - K = 20

In [ ]:
# algorithme des K-means k = 20

nb_clusters = 20
X = [[x,y] for (x,y) in zip(s_centroids_dist.x, s_centroids_dist.y)]
kmeans = KMeans(n_clusters = nb_clusters, random_state=0).fit(X)#, sample_weight=df_centroids['weight'].values)
#gdf = gpd.GeoDataFrame(df_centroids, geometry=s_centroids)

gdf_areas['cluster'] = kmeans.labels_
df_centroids['cluster'] = kmeans.labels_

gdf_KMeans_20 = gdf_areas.dissolve(by='cluster')

In [ ]:
# indicateur

inertia = kmeans.inertia_
silhouette = silhouette_score(X, kmeans.labels_)
davies = davies_bouldin_score(X, kmeans.labels_)
calinski = calinski_harabasz_score(X, kmeans.labels_)

# Affichage propre
df_metrics = pd.DataFrame({
    "Indicateur": [
        "Inertie (within-cluster SSE)",
        "Silhouette score",
        "Davies-Bouldin index",
        "Calinski-Harabasz index"
    ],
    "Valeur": [
        inertia,
        silhouette,
        davies,
        calinski
    ]
})

print(df_metrics)

In [ ]:
# affichage des K-means avec k = 20

list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]
db_access = pd.DataFrame({'long':list_long, 'lat':list_lat})

gdf_access = gpd.GeoDataFrame(db_access, geometry = gpd.points_from_xy(list_long, list_lat))
gdf_access.crs = 'EPSG:4326'
gdf_access = gdf_access.to_crs('EPSG:3857')
gdf_access_dist = gdf_access.to_crs('EPSG:28992')

nb_points = len(db_access)

gdf_access['closest_centroid'] = np.zeros(nb_points)
gdf_access['i_closest_centroid'] = np.zeros(nb_points)
for i,row in gdf_access_dist.iterrows():
    x=row.geometry.x
    y=row.geometry.y
    dists = [(x-x_)**2+(y-y_)**2 for (x_,y_) in zip(s_centroids_dist.x, s_centroids_dist.y)]
    centroid_id = df_centroids['id'][np.argmin(dists)]
    gdf_access.loc[i, 'i_closest_centroid'] = np.argmin(dists)
    gdf_access.loc[i, 'closest_centroid'] = centroid_id


fig, ax=plt.subplots(figsize=(20,20))
#gdf_clusters = gdf_clusters.to_crs('EPSG:3857')
gdf_areas.centroid.plot(ax=ax, color='b')
gdf_KMeans_20.centroid.plot(ax=ax, color = 'r')
gdf_KMeans_20.reset_index().plot(
    ax=ax, column='cluster', cmap='rainbow', edgecolor='black', alpha=0.5, zorder=1)
#gdf_edges[gdf_edges.FuncClass<5].plot(ax=ax, color='grey')
gdf_access.plot(ax=ax, color='magenta', markersize=100)
cx.add_basemap(ax=ax, url=cx.providers.OpenStreetMap.Mapnik)
for c in range(nb_clusters):
    x = gdf_KMeans_20.centroid.loc[c].x
    y = gdf_KMeans_20.centroid.loc[c].y
    ax.text(x,y,str(c), zorder=2, size=40)
for p in range(nb_points):
    x = gdf_access.centroid.loc[p].x
    y = gdf_access.centroid.loc[p].y
    t = ax.text(x,y,str(p), zorder=2, size=40, color='m')
    t.set_bbox(dict(facecolor='w', alpha=0.5, edgecolor='w'))
    i_cen_id = int(gdf_access.loc[p, 'i_closest_centroid'])
    cl_cen = gdf_areas.centroid[i_cen_id]
    ax.plot(cl_cen.x, cl_cen.y, 'kd', alpha=0.5, markersize=20)

In [ ]:
# création de la classification OD

# Polygones des clusters (issus du dissolve)
gdf_polygons_K20 = gdf_KMeans_20.reset_index()[['cluster', 'geometry']]
gdf_polygons_K20 = gdf_polygons_K20.rename(columns={'cluster': 'CLUSTER_ID'})

gdf_polygons_K20 = gpd.GeoDataFrame(gdf_polygons_K20, geometry='geometry', crs = 'EPSG:3857')
gdf_polygons_K20 = gdf_polygons_K20.to_crs("epsg:32631")

gdf_points = gpd.GeoDataFrame(
    db_access.copy(), 
    geometry=gpd.points_from_xy(db_access.long, db_access.lat),
    crs="EPSG:4326"
)

# Reprojection dans le même CRS que les polygones
gdf_points = gdf_points.to_crs(gdf_polygons_K20.crs)

# Ajout identifiants
gdf_points = gdf_points.reset_index().rename(columns={'index': 'CLUSTER_ID'})
gdf_points = gdf_points[['CLUSTER_ID', 'geometry']]


clusters_OD_K20 = pd.concat([gdf_polygons_K20, gdf_points], ignore_index=True)
clusters_OD_K20 = gpd.GeoDataFrame(clusters_OD_K20, geometry='geometry', crs=gdf_polygons_K20.crs)

clusters_OD_K20 = clusters_OD_K20.merge(
    clusters_OD_K20,
    how="cross",
    suffixes=("_origin", "_dest")
)


#---------------------------------------
# 3. Colonnes demandées
#---------------------------------------
clusters_OD_K20 = clusters_OD_K20.rename(columns={
    "CLUSTER_ID_origin": "ORIGIN_CLUSTER",
    "CLUSTER_ID_dest": "DESTINATION_CLUSTER",
    "geometry_origin": "ORIGIN",
    "geometry_dest": "DESTINATION"
})

clusters_OD_K20["CLUSTER_ID"] = clusters_OD_K20.index


#---------------------------------------
# 4. Colonnes finales
#---------------------------------------
clusters_OD_K20 = clusters_OD_K20[[
    "CLUSTER_ID",
    "ORIGIN_CLUSTER",
    "DESTINATION_CLUSTER",
    "ORIGIN",
    "DESTINATION"
]]

clusters_OD_K20

clusters_OD_K20.to_csv("../../data/classification/KMeans/Classification_KMeans-20.csv", sep=';', index=False)

## 4 - Association

Dans cette partie, les découpages précédemment produits sont associés aux demandes. Les classes de chaque classification sont ensuite ordonnées par ordre décroissant d'importance d'effectif.

### 4.1 - Matsim_7-9_Calibration

In [ ]:
# association classification amsterdam

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

buffer = ClassificationUtils.associate(chemin_matsim_calibration,classification_amsterdam,crs,False)
matsim_calibration["CLUSTER_ID"] = buffer

matsim_calibration.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_Amsterdam.csv", sep=';', index=False)

In [ ]:
# association classification K7

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-7.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

buffer = ClassificationUtils.associate(chemin_matsim_calibration,classification_amsterdam,crs,False)
matsim_calibration["CLUSTER_ID"] = buffer

matsim_calibration.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K7.csv", sep=';', index=False)

In [ ]:
# association classification K10

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-10.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

buffer = ClassificationUtils.associate(chemin_matsim_calibration,classification_amsterdam,crs,False)
matsim_calibration["CLUSTER_ID"] = buffer

matsim_calibration.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K10.csv", sep=';', index=False)

In [ ]:
# association classification K15

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-15.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

buffer = ClassificationUtils.associate(chemin_matsim_calibration,classification_amsterdam,crs,False)
matsim_calibration["CLUSTER_ID"] = buffer

matsim_calibration.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K15.csv", sep=';', index=False)

In [ ]:
# association classification K20

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-20.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

buffer = ClassificationUtils.associate(chemin_matsim_calibration,classification_amsterdam,crs,False)
matsim_calibration["CLUSTER_ID"] = buffer

matsim_calibration.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K20.csv", sep=';', index=False)

### 4.2 - Amsterdam_7-9_Original

In [ ]:
# association classification amsterdam

chemin_amsterdam_original = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
crs = "epsg:32631"
amsterdam_original = pd.read_csv(chemin_amsterdam_original, sep=';')

buffer = ClassificationUtils.associate(chemin_amsterdam_original,classification_amsterdam,crs,False)
amsterdam_original["CLUSTER_ID"] = buffer

amsterdam_original.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_Amsterdam.csv", sep=';', index=False)

In [ ]:
# association classification K7

chemin_amsterdam_original = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-7.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_amsterdam_original, sep=';')

buffer = ClassificationUtils.associate(chemin_amsterdam_original,classification_amsterdam,crs,False)
amsterdam_original["CLUSTER_ID"] = buffer

amsterdam_original.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K7.csv", sep=';', index=False)

In [ ]:
# association classification K10

chemin_amsterdam_original = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-10.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_amsterdam_original, sep=';')

buffer = ClassificationUtils.associate(chemin_amsterdam_original,classification_amsterdam,crs,False)
amsterdam_original["CLUSTER_ID"] = buffer

amsterdam_original.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K10.csv", sep=';', index=False)

In [ ]:
# association classification K15

chemin_amsterdam_original = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
classification_amsterdam = "../../data/classification/KMeans/Classification_KMeans-15.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_amsterdam_original, sep=';')

buffer = ClassificationUtils.associate(chemin_amsterdam_original,classification_amsterdam,crs,False)
amsterdam_original["CLUSTER_ID"] = buffer

amsterdam_original.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K15.csv", sep=';', index=False)

In [ ]:
# association classification K20

chemin_amsterdam_original = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
classification_amsterdam = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
crs = "epsg:32631"
matsim_calibration = pd.read_csv(chemin_amsterdam_original, sep=';')

buffer = ClassificationUtils.associate(chemin_amsterdam_original,classification_amsterdam,crs,False)
amsterdam_original["CLUSTER_ID"] = buffer

amsterdam_original.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K20.csv", sep=';', index=False)

## 5 - Classement

Ici sont associées les classifications spatiales aux deux demandes puis sont classées les classes  de chacune d'entres elles en fonction de leur effectif.

### 5.1 - Matsim_7-9_Calibration

In [ ]:
# classement classification amsterdam

chemin_demande_classification = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_Amsterdam.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

matsim_classement_classification = ClassificationUtils.ordered_classification(buffer)
matsim_classement_classification.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_Amsterdam.csv", sep=';', index=False)

In [ ]:
# classement classification K7

chemin_demande_classification = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K7.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

matsim_classement_classification = ClassificationUtils.ordered_classification(buffer)
matsim_classement_classification.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K7.csv", sep=';', index=False)

In [ ]:
# classement classification K10

chemin_demande_classification = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K10.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

matsim_classement_classification = ClassificationUtils.ordered_classification(buffer)
matsim_classement_classification.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K10.csv", sep=';', index=False)

In [ ]:
# classement classification K15

chemin_demande_classification = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K15.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

matsim_classement_classification = ClassificationUtils.ordered_classification(buffer)
matsim_classement_classification.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K15.csv", sep=';', index=False)

In [ ]:
# classement classification K20

chemin_demande_classification = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Classification_K20.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

matsim_classement_classification = ClassificationUtils.ordered_classification(buffer)
matsim_classement_classification.to_csv("../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K20.csv", sep=';', index=False)

### 5.2 - Amsterdam_7-9_Original

In [ ]:
# classement classification amsterdam

chemin_demande_classification = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_Amsterdam.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

amsterdam_classement_classification = ClassificationUtils.ordered_classification(buffer)
amsterdam_classement_classification.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_Amsterdam.csv", sep=';', index=False)

In [ ]:
# classement classification K7

chemin_demande_classification = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K7.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

amsterdam_classement_classification = ClassificationUtils.ordered_classification(buffer)
amsterdam_classement_classification.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K7.csv", sep=';', index=False)

In [ ]:
# classement classification K10

chemin_demande_classification = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K10.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

amsterdam_classement_classification = ClassificationUtils.ordered_classification(buffer)
amsterdam_classement_classification.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K10.csv", sep=';', index=False)

In [ ]:
# classement classification K15

chemin_demande_classification = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K15.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

amsterdam_classement_classification = ClassificationUtils.ordered_classification(buffer)
amsterdam_classement_classification.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K15.csv", sep=';', index=False)

In [ ]:
# classement classification K20

chemin_demande_classification = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Classification_K20.csv"
buffer = pd.read_csv(chemin_demande_classification, sep=';')

amsterdam_classement_classification = ClassificationUtils.ordered_classification(buffer)
amsterdam_classement_classification.to_csv("../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K20.csv", sep=';', index=False)

## 6 - Analyse

Dans cette partie sont comparées les regroupements pour chaque méthode entre type de demande.

In [ ]:
# fonctions

def plot_cluster_maps_plotly(
    path_df_left,
    path_df_right,
    path_geom,
    crs,
    title_left,
    title_right
):
    import pandas as pd
    import geopandas as gpd
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    from shapely import wkt

    # ---------- utilitaires ----------
    def to_centroid(geom):
        return geom.centroid if geom.geom_type != "Point" else geom

    def spread_points_grid(points, offset=0.009):
        counts = {}
        new_coords = []
        for x, y in points:
            key = (round(x, 6), round(y, 6))
            n = counts.get(key, 0)
            dx = offset * (n % 3 - 1)
            dy = offset * (n // 3 - 1)
            new_coords.append((x + dx, y + dy))
            counts[key] = n + 1
        return new_coords

    # ---------- lecture ----------
    df_left = pd.read_csv(path_df_left, sep=';').head(10)
    df_right = pd.read_csv(path_df_right, sep=';').head(10)
    df_geom = pd.read_csv(path_geom, sep=';')

    left = df_left.merge(df_geom, on="CLUSTER_ID")
    right = df_right.merge(df_geom, on="CLUSTER_ID")


    def prepare(df, col):
        gdf = gpd.GeoDataFrame(
            df,
            geometry=df[col].apply(wkt.loads),
            crs=crs
        )
        gdf["geometry"] = gdf["geometry"].apply(to_centroid)
        return gdf.to_crs(epsg=4326)

    gdf_lo = prepare(left, "ORIGIN")
    gdf_ld = prepare(left, "DESTINATION")
    gdf_ro = prepare(right, "ORIGIN")
    gdf_rd = prepare(right, "DESTINATION")

    # ---------- assigner couleur par CLUSTER_ID ----------
    all_cluster_ids = pd.concat([left["CLUSTER_ID"], right["CLUSTER_ID"]]).drop_duplicates().reset_index(drop=True)
    colors = px.colors.qualitative.Dark24
    color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(all_cluster_ids)}

    # ---------- figure ----------
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "map"}, {"type": "map"}]],
        subplot_titles=[title_left, title_right]
    )

    def add_map(gdf_o, gdf_d, df_orig, col):
        all_pts = [(p.x, p.y) for p in gdf_o.geometry] + [(p.x, p.y) for p in gdf_d.geometry]
        all_pts_spread = spread_points_grid(all_pts)
        pts_o = all_pts_spread[:len(gdf_o)]
        pts_d = all_pts_spread[len(gdf_o):]

        for i, (_, row) in enumerate(gdf_o.iterrows()):
            cid = row.CLUSTER_ID
            count = row.COUNT
            color = color_map[cid]
            rank = df_orig[df_orig["CLUSTER_ID"] == cid].index[0] + 1

            xo, yo = pts_o[i]
            xd, yd = pts_d[i]

            # ORIGIN
            fig.add_trace(
                go.Scattermap(
                    lon=[xo],
                    lat=[yo],
                    mode="markers+text",
                    marker=dict(size=10, color=color, opacity=0.6, symbol="circle"),
                    text=[str(rank)],
                    textposition="middle center",
                    hovertemplate=f"<b>Cluster {cid}</b><br>COUNT: {count}<br>TYPE: ORIGIN<extra></extra>",
                    showlegend=False
                ),
                row=1, col=col
            )

            # DESTINATION
            fig.add_trace(
                go.Scattermap(
                    lon=[xd],
                    lat=[yd],
                    mode="markers+text",
                    marker=dict(size=14, color=color, opacity=0.6, symbol="circle"),
                    text=[str(rank)],
                    textposition="middle center",
                    hovertemplate=f"<b>Cluster {cid}</b><br>COUNT: {count}<br>TYPE: DESTINATION<extra></extra>",
                    name=f"Cluster {cid} (COUNT={count})"
                ),
                row=1, col=col
            )

    add_map(gdf_lo, gdf_ld, df_left, col=1)
    add_map(gdf_ro, gdf_rd, df_right, col=2)

    # ---------- centre et zoom ----------
    all_points = pd.concat([gdf_lo.geometry, gdf_ld.geometry, gdf_ro.geometry, gdf_rd.geometry])
    minx, miny, maxx, maxy = all_points.total_bounds
    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2
    zoom = 10.5

    fig.update_layout(
        map1=dict(center={"lat": center_lat, "lon": center_lon}, zoom=zoom),
        map2=dict(center={"lat": center_lat, "lon": center_lon}, zoom=zoom),
        width=1100,
        height=500,
        margin=dict(l=10, r=10, t=60, b=10),
        legend_title_text="Clusters"
    )

    return fig



### 6.1 - Classification Amsterdam

In [ ]:
# difference demande Matsim_7-9 et demande AmsterdamCity_7-9

path_df_left = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_Amsterdam.csv"
path_df_right = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_Amsterdam.csv"
path_geom = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"
crs = "epsg:32631"
title_left = "matsim"
title_right="amsterdam"

plot_cluster_maps_plotly(path_df_left,
    path_df_right,
    path_geom,
    crs = crs,
    title_left=title_left,
    title_right=title_right          
)

### 6.2 - Classification K7

In [ ]:
# difference demande Matsim_7-9 et demande AmsterdamCity_7-9

path_df_left = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K7.csv"
path_df_right = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K7.csv"
path_geom = "../../data/classification/KMeans/Classification_KMeans-7.csv"
crs = "epsg:32631"
title_left = "matsim"
title_right="amsterdam"

plot_cluster_maps_plotly(path_df_left,
    path_df_right,
    path_geom,
    crs = crs,
    title_left=title_left,
    title_right=title_right          
)

### 6.3 - Classification K10

In [ ]:
# difference demande Matsim_7-9 et demande AmsterdamCity_7-9

path_df_left = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K10.csv"
path_df_right = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K10.csv"
path_geom = "../../data/classification/KMeans/Classification_KMeans-10.csv"
crs = "epsg:32631"
title_left = "matsim"
title_right="amsterdam"

plot_cluster_maps_plotly(path_df_left,
    path_df_right,
    path_geom,
    crs = crs,
    title_left=title_left,
    title_right=title_right          
)

### 6.4 - Classification K15

In [ ]:
# difference demande Matsim_7-9 et demande AmsterdamCity_7-9

path_df_left = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K15.csv"
path_df_right = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K15.csv"
path_geom = "../../data/classification/KMeans/Classification_KMeans-15.csv"
crs = "epsg:32631"
title_left = "matsim"
title_right="amsterdam"

plot_cluster_maps_plotly(path_df_left,
    path_df_right,
    path_geom,
    crs = crs,
    title_left=title_left,
    title_right=title_right          
)

### 6.5 - Classification K20

In [ ]:
# difference demande Matsim_7-9 et demande AmsterdamCity_7-9

path_df_left = "../../data/classification/sample/Matsim_7-9_Calibration_Original__Ranking_K20.csv"
path_df_right = "../../data/classification/sample/AmsterdamCity_7-9_Original_Original__Ranking_K20.csv"
path_geom = "../../data/classification/KMeans/Classification_KMeans-20.csv"
crs = "epsg:32631"
title_left = "matsim"
title_right="amsterdam"

plot_cluster_maps_plotly(path_df_left,
    path_df_right,
    path_geom,
    crs = crs,
    title_left=title_left,
    title_right=title_right          
)

## 7 - Bilan